# **Schnorr Digital Signature Algorithm (EC SSA)**

# Setup

btclib is needed: let's install/update it and import straight away some of its functions

In [1]:
# Copyright (c) The btclib developers
# Distributed under the MIT software license, see the accompanying
# LICENSE file or https://opensource.org/license/mit for the full text.

!pip install --upgrade btclib

from hashlib import sha256 as hf

from btclib.curves.curve import double_mult_var, mult
from btclib.curves.curve import secp256k1 as ec

For this exercise we use secp256k1 as elliptic curve and SHA256 as hash function:

In [2]:
print(ec)

Curve
 p   = FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFE FFFFFC2F
 a   = 0
 b   = 7
 x_G = 79BE667E F9DCBBAC 55A06295 CE870B07 029BFCDB 2DCE28D9 59F2815B 16F81798
 y_G = 483ADA77 26A3C465 5DA4FBFC 0E1108A8 FD17B448 A6855419 9C47D08F FB10D4B8
 n   = FFFFFFFF FFFFFFFF FFFFFFFF FFFFFFFE BAAEDCE6 AF48A03B BFD25E8C D0364141
 cofactor = 1


BIP340 fixes both, and fixes them to match: SHA256 for the hash function and secp256k1 for the curve, so the digest is exactly as wide as the order of the group. ECDSA left the pair free and the overall security to the smaller of the two

In [3]:
print(hf().digest_size)
print(ec.n_size)

32
32


# **Digital Signature Protocol**

## 1. Key generation

Private key (generated elsewhere, a fixed value here):

In [4]:
q = 0x18E14A7B6A307F426A94F8114701E7C8E774E7F9A47E2C2035DB29A206321725
assert 0 < q < ec.n, "Invalid private key"
print("q:", q)
print("Hex(q):", hex(q))

q: 11253563012059685825953619222107823549092147699031672238385790369351542642469
Hex(q): 0x18e14a7b6a307f426a94f8114701e7c8e774e7f9a47e2c2035db29a206321725


Corresponding Public Key.

A BIP340 public key is *x-only*: the y coordinate is not transmitted, and the reader takes the even one of the two. So the private key is used in whichever of `q` and `n - q` puts its public key on the even root, which is a choice about the key and not about any signature made with it.

In [5]:
Q = mult(q)
if Q[1] % 2:
    q = ec.n - q
    Q = (Q[0], ec.p - Q[1])
assert Q[1] % 2 == 0

print("PubKey:", hex(Q[0]))

PubKey: 0x50863ad64a87ae8a2fe83c1af1a8403cb53f53e486d8511dad8a04887e5b2352


## Signature (SSA)

### Message
Message to be signed and its hash value:

In [6]:
msg = "Paolo is afraid of ephemeral random numbers"

# the signed message is 32 bytes and stays bytes: unlike ECDSA, BIP340
# never turns it into an integer modulo ec.n -- it goes into the
# challenge hash as it is
m = hf(msg.encode()).digest()
print("m:", m.hex())

m: 9788fd27b3aafd1bd1591a1158ce2d8bdc37ab4040dddb64e64d17616e69ce2b


### Deterministic Ephemeral key
The Ephemeral key k must be kept secret and never reused
A good choice is to use a deterministic key: 

`k = hf(q||m)` 

different for each msg, private because of q.

BIP340 specifies a nonce of its own -- `tagged_hash("BIP0340/nonce", t || x_Q || m)`, with `t` the private key masked by auxiliary randomness -- and a signer following the specification uses that one. The simpler form is kept here because the section after next breaks it on purpose, and what it breaks has to be visible.

In [7]:
k_bytes = hf(q.to_bytes(32, 'big') + m).digest()
k = int.from_bytes(k_bytes, 'big') % ec.n
assert k != 0
print("eph k:", hex(k))

eph k: 0x3fa3ccf6c168482533f1fa066650704546e56f0bf15fbfb3b4bc51f404e19ee7


### The challenge

This is where Schnorr parts company with ECDSA. ECDSA multiplies the hash and the private key together through the inverse of the ephemeral key; Schnorr adds, and what it adds is a *challenge* over everything the verifier will already have:

`c = int(tagged_hash("BIP0340/challenge", x_K || x_Q || m)) mod n`

BIP340 hashes with a tagged hash, so a digest computed for one purpose cannot be replayed as another:

`tagged_hash(tag, x) = hf(hf(tag) || hf(tag) || x)`

In [8]:
def tagged_hash(tag, x):
    """Return BIP340's tagged hash of x under tag."""
    t = hf(tag.encode()).digest()
    return hf(t + t + x).digest()


def challenge(x_K, x_Q, m):
    """Return the challenge over the two x coordinates and the message."""
    t = tagged_hash("BIP0340/challenge",
                    x_K.to_bytes(32, 'big') + x_Q.to_bytes(32, 'big') + m)
    return int.from_bytes(t, 'big') % ec.n

### Signature Algorithm

In [9]:
K = mult(k)
# x_K is all the verifier gets, so K is the even-y point of the two, and
# an ephemeral key landing on the odd one is negated -- the same choice
# made for the public key above, made again for each signature
if K[1] % 2:
    k = ec.n - k
    K = (K[0], ec.p - K[1])

r = K[0]
c = challenge(r, Q[0], m)
assert c != 0

s = (k + c*q) % ec.n

print("r:", hex(r))
print("c:", hex(c))
print("s:", hex(s))

r: 0xf91a63e7574b8a7ea99cc8999456e8044b9d1cb05a3ec25c3e8886cee3ed0142
c: 0x99baa84e841e36a7bcc3f7e9dab5d0cfc71c236dcc902be60215e1cda39504c2
s: 0x4c699580c8d153d06b57ae31a74030ea32a2d84505c6ba2375f67283c3fb4624


## Signature verification (SSA)

The signer computed `s = k + c*q`. Multiply both sides by G and the ephemeral point comes back out: `s*G - c*Q = k*G = K`. No inverse is taken anywhere.

In [10]:
K_v = double_mult_var(-c, Q, s, ec.G)
print(K_v[0] == r and K_v[1] % 2 == 0)

True


## Malleated Signature

An ECDSA signature has a twin: `(r, n - s)` verifies wherever `(r, s)` does, because that verification recovers a point from `r` and compares x coordinates only, and negating `s` negates a y nobody looks at. Bitcoin's low-s rule exists to pick one of the two.

The check just above is a linear equation in `s`, so a different `s` gives a different point:

In [11]:
sm = ec.n - s
print("     r:", hex(r))
print("   *sm:", hex(sm))

K_m = double_mult_var(-c, Q, sm, ec.G)
print(K_m[0] == r and K_m[1] % 2 == 0)

     r: 0xf91a63e7574b8a7ea99cc8999456e8044b9d1cb05a3ec25c3e8886cee3ed0142
   *sm: 0xb3966a7f372eac2f94a851ce58bfcf14880c04a1a981e61849dbec090c3afb1d
False


## Humongous mistake

### Message
Message to be signed and its hash value:

In [12]:
msg2 = "and Paolo is right to be afraid"

m2 = hf(msg2.encode()).digest()
print("m2:", m2.hex())

m2: 7adb91982ec03ef87efcae7f0199aefa231d8855e0bd03319460e58c0bd18049


### The mistake 
Reuse the same ephemeral deterministic key as the previous message:

In [13]:
k2 = k #very bad! Never reuse the same ephemeral key!!!
# k is not the k printed further up: signing negated it, K having come
# out on the odd root. Reusing it is the same mistake either way
print("eph k :", hex(k))
print("eph k2:", hex(k2))

eph k : 0xc05c33093e97b7dacc0e05f999af8fb973c96ddabde8e0880b160c98cb54a25a
eph k2: 0xc05c33093e97b7dacc0e05f999af8fb973c96ddabde8e0880b160c98cb54a25a


### Signature Algorithm

In [14]:
K2 = mult(k2)
if K2[1] % 2:
    k2 = ec.n - k2
    K2 = (K2[0], ec.p - K2[1])

r2 = K2[0]
# r2 comes out equal to r, the same ephemeral key giving back the same
# point: the mistake is visible to anybody holding the two signatures,
# before any arithmetic is done on them
c2 = challenge(r2, Q[0], m2)
assert c2 != 0

s2 = (k2 + c2*q) % ec.n

print("r2:", hex(r2))
print("c2:", hex(c2))
print("s2:", hex(s2))

r2: 0xf91a63e7574b8a7ea99cc8999456e8044b9d1cb05a3ec25c3e8886cee3ed0142
c2: 0x8e20739f6cf64d61d92e02c8d209e6da041ac694cb4f0bde92772166cae1b786
s2: 0x320570a86cf4787c364a452116ae302ac410eeda34a1d02de42bb185b557a794


### Signature verification

In [15]:
K_v = double_mult_var(-c2, Q, s2, ec.G)
print(K_v[0] == r2 and K_v[1] % 2 == 0)

True


### Exercise
Because of this mistake is now possible to calculate the private key from the 2 signatures. 

`from btclib.number_theory import mod_inv` is the inverse you will need.

In [16]:
# forget k, k2, q
k=k2=q=0

# solve the problem of calculating q
# using only r, s, c, s2, and c2:
# q =

print(hex(q))
print(mult(q) == Q) # check it is the correct private key

0x0
False
